In [ ]:
import os
from mlflow import MlflowClient
import mlflow
import json

experiment_id = "466819623607489962"

# Set the tracking URI using environment variable or programmatically
# Option 1: Set via environment variable (uncomment if you want to use this method)
# os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"

# Option 2: Set programmatically (recommended for notebooks/scripts)
mlflow.set_tracking_uri("http://localhost:5050")

client = MlflowClient()
traces = client.search_traces(experiment_ids=[experiment_id])


In [7]:
print(traces)

[Trace(trace_id=64d74bd440534bbc81960ee1ce07c5f4), Trace(trace_id=e56e9dddc3ab42bbbdfe4d5686749393), Trace(trace_id=c686e78f4903427f8d28cd2032bfdd5b), Trace(trace_id=12bc990d3f3f4eaa881e0b65b09b47da)]


In [10]:
# To handle keys that are still JSON strings, we can recursively parse any value that looks like a JSON string.
def try_json_load(val):
    if isinstance(val, str):
        try:
            loaded = json.loads(val)
            # If loaded is a dict or list, recursively process it
            if isinstance(loaded, dict):
                return {k: try_json_load(v) for k, v in loaded.items()}
            elif isinstance(loaded, list):
                return [try_json_load(x) for x in loaded]
            else:
                return loaded
        except Exception:
            return val
    elif isinstance(val, dict):
        return {k: try_json_load(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [try_json_load(x) for x in val]
    else:
        return val

for i, trace in enumerate(traces):
    trace_dict = json.loads(trace.to_json())
    print(trace_dict)
    trace_dict = try_json_load(trace_dict)
    # Get timestamp from info.request_time if available
    timestamp = None
    request_time = (
        trace_dict.get("info", {}).get("request_time")
        if "info" in trace_dict and "request_time" in trace_dict["info"]
        else None
    )
    if request_time is not None:
        # Optionally, you could format the request_time string if needed
        # For now, just use the raw string, replacing characters not allowed in filenames
        safe_request_time = (
            request_time.replace(":", "-").replace("T", "_").replace("Z", "")
        )
        timestamp = safe_request_time
    # Fallback to index if no timestamp found
    filename = f"trace_{timestamp if timestamp is not None else i}.json"
    with open(filename, "w") as f:
        json.dump(trace_dict, f, indent=2, default=str)

{'info': {'trace_id': '64d74bd440534bbc81960ee1ce07c5f4', 'trace_location': {'type': 'MLFLOW_EXPERIMENT', 'mlflow_experiment': {'experiment_id': '466819623607489962'}}, 'request_time': '2025-08-04T02:49:20.887Z', 'state': 'OK', 'trace_metadata': {'mlflow.trace_schema.version': '3', 'mlflow.trace.tokenUsage': '{"input_tokens": 8691, "output_tokens": 599, "total_tokens": 9290}', 'mlflow.traceInputs': '{"messages": [{"content": "get the economic trend news from the web for q1 2024 and how it may affect our sale in q1 2024!", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null, "example": false}]}', 'mlflow.source.type': 'LOCAL', 'mlflow.source.git.branch': 'main', 'mlflow.source.name': '/Users/iwsono/Documents/Workspace/SchoolProject/!FinalProject/super-store-agent/server/.venv/bin/flask', 'mlflow.source.git.commit': 'f30892972ba2e967a628a7617863c09c06a37264', 'mlflow.user': 'iwsono', 'mlflow.source.git.repoURL': 'https://github.com/w4rlock999/super